# 🏴‍☠️ Pirate Web Summarizer using Ollama & Python

This notebook demonstrates how to build an AI-powered tool that fetches the text content from any website and summarizes it using an LLM running locally via Ollama. 

**What this notebook does:**
1. **Web Scraping:** Fetches the raw text content of a target website.
2. **LLM Integration:** Connects to a local Ollama instance.
3. **Persona-driven Summarization:** Uses prompt engineering to adopt a Caribbean Pirate persona, summarizing the core content of the website while ignoring navigation menus and boilerplate text.
4. **Markdown Rendering:** Displays the final summary cleanly using IPython's Markdown display features.

In [ ]:
# imports

import os
from dotenv import load_dotenv
from advanced_scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import asyncio

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)

api_key = os.getenv('OLLAMA_API_KEY')
base_url = os.getenv('OLLAMA_BASE_URL')
model = 'mistral'

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
# Define our system prompt
system_prompt = """
You are a seasoned Caribbean pirate tasked with scouting digital territories. I will provide you with the text extracted from a website.

Your objective is to analyze the text and provide a short, engaging summary of the core information in your pirate persona.

Guidelines:
1. Ignore the Clutter: Disregard all navigation links, menus, sidebars, footers, and boilerplate text. Focus only on the main treasure (the primary content).
2. Pirate Persona: Heavily utilize pirate slang, nautical terms, and a rugged tone (e.g., "Ahoy," "booty," "shiver me timbers," "scallywag").
3. Keep it Brief: Keep your summary concise, around 2 to 4 sentences.
4. Formatting: Respond strictly in standard Markdown.

CRITICAL INSTRUCTION: Do NOT wrap your response in a Markdown code block (e.g., no ```markdown ... ```). Output only the raw Markdown text.
"""

In [ ]:
# Define our user prompt

user_prompt_prefix = """
Here be the text we scouted from the target website. 

Look through it and tell me:
1. What manner of place is this website? (A short overview)
2. Is there any fresh news or announcements we should care about?

Target Website Text:
--------------------

"""

In [ ]:
ollama = OpenAI(base_url=base_url,api_key=api_key)

In [ ]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
# And now: call the LLM. You will get very familiar with this!
async def summarize(url):
    website = await fetch_website_contents(url)
    try:
        response = await asyncio.to_thread(
            ollama.chat.completions.create,
            model=model,
            messages=messages_for(website)
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"LLM ERROR: {e}"

In [ ]:
# A function to display this nicely in the output, using markdown

async def display_summary(url):
    summary = await summarize(url)
    display(Markdown(summary))

In [ ]:
URL=""
await display_summary(URL)